In [ ]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2
# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from pathlib import Path
from typing import List, Optional, Dict, Any

from src.logging_config import setup_logging, get_logger
from src.config import Settings, get_settings

setup_logging()
logger = get_logger(__name__)

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from langchain_core.documents import Document
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.util import generate_uuid5
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
import base64
from weaviate.classes.query import Filter
from caption_utils import attach_captions
from langchain.text_splitter import RecursiveCharacterTextSplitter


class TestIngestService:
    """
    Service class for document ingestion operations.
    Designed for use in API endpoints.
    """
    
    def __init__(self, settings: Settings | None = None):
        self.settings = settings or get_settings()
        self._client: Optional[weaviate.WeaviateClient] = None

    def __enter__(self):
        """Context manager entry."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit - ensures cleanup."""
        self.close()
        return False

    def close(self):
        """Close Weaviate client connection."""
        if self._client:
            self._client.close()
            self._client = None
    
    def _get_client(self) -> weaviate.WeaviateClient:
        """Get or create Weaviate client using config settings."""
        if self._client is None or not self._client.is_connected():
            self._client = weaviate.connect_to_local(
                host=self.settings.weaviate_host,
                port=self.settings.weaviate_http_port,
                grpc_port=self.settings.weaviate_grpc_port
            )
        return self._client
    
    def _summarize_image(self, chunk: dict) -> dict:
        """Summarize an image chunk using LLM vision."""
        try:

            caption = chunk['metadata']['caption']

            llm = ChatGoogleGenerativeAI(
                model="gemini-2.0-flash-lite",
                temperature=0.1,
                google_api_key=self.settings.gemini_api_key
            )
            system_message = SystemMessage(content=f"""You are a document analyst preparing content for a semantic search index.

    You are given:
    1. An image extracted from a document
    2. The image's caption: "{caption}"

    Your task is to write a concise, information-dense summary of this image that will be used as the text representation for vector search retrieval.

    **Instructions:**
    - Use the caption as the primary context anchor — it tells you what the image is about.
    - Describe what the image actually shows: diagrams, charts, tables, architectures, equations, workflows, relationships, etc.
    - Extract ALL specific entities: names, labels, numbers, metrics, axis values, legends, annotations, and technical terms visible in the image.
    - Preserve the original terminology exactly as it appears (do not paraphrase technical terms).
    - State the key takeaway or insight the image conveys.
    - If the image contains comparisons or trends, describe them explicitly (e.g., "X outperforms Y by Z%").
    - Write in plain, factual sentences. Do NOT use bullet points or markdown formatting.
    - Do NOT say "the image shows" or "this figure illustrates" — just state the information directly.
    - Keep the summary between 2-5 sentences, prioritizing information density over length.""")

            # Process image to base64
            try: 
                content = []
                img_path = chunk['metadata']['image_path']
                with open(img_path, 'rb') as f:
                    img_base64 = base64.b64encode(f.read()).decode('utf-8')
                    content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                    })
            except Exception as e:
                logger.warning(f"Warning: Could not load image {img_path}: {e}")
            
            # Create messages and invoke LLM
            human_message = HumanMessage(content=content)
            response = llm.invoke([system_message, human_message])

            summary = response.content.strip()
            chunk['text'] = f"{caption}\n\n{summary}"
            
            logger.info(f"Generated summary for image: {img_path}")

            return chunk 

        except Exception as e:
            logger.error(f"Error generating summaries: {e}")
            raise e


    def preprocess_documents(self, file_name: str, collection_name: str):
        try:
            # Document file path (per-collection folder)
            file_path = self.settings.base_dir / "data" / "raw" / collection_name / file_name

            # Folder store processed images and tables (per-collection folder)
            processed_folder_path = self.settings.base_dir / "data" / "processed" / collection_name / file_name.split('.')[0]

            # Extract elements from PDF
            elements = partition_pdf(
                filename=file_path,
                strategy="hi_res",
                # hi_res_model_name="yolox_quantized",   # Faster model -> less accuracy extract elements
                pdf_image_dpi=150,                      # Lower resolution (before 200)
                ocr_mode="individual_blocks",           # Skip full-page OCR (not scanned PDF)
                extract_image_block_types=["Image", "Table"],
                extract_image_block_to_payload=False,
                extract_image_block_output_dir=processed_folder_path
            )
            
            # remove insignificant elements
            significant_elements = [
                ele for ele in elements 
                if ele.to_dict().get('type') not in ['UncategorizedText', 'Header'] 
                and len(ele.to_dict().get('text', '')) > 2
            ]

            # attach captions (modifies Image/Table text in-place, removes caption elements)
            significant_elements, message = attach_captions(significant_elements)
            logger.info(message)
            # Split into text vs multimodal (also summarize images chunks)
            text_elements = []
            multimodal_documents = []
            
            for ele in significant_elements:
                source = ele.to_dict()['metadata'].get('file_directory', '') + '/' + ele.to_dict()['metadata'].get('filename', '')
                ele.metadata.source = source

                ele_dict = ele.to_dict()
                doc_type = ele_dict.get('type', '')

                if doc_type in ['Image', 'Table']:
                    if doc_type == 'Image':
                        ele_dict = self._summarize_image(ele_dict)
                    # Tranform to Document for tables, images
                    document = Document(
                        page_content=ele_dict.get('text', ''),
                        metadata={
                            'type': ele_dict.get('type', ''),
                            'id': ele_dict.get('element_id', ''),
                            'caption': ele_dict['metadata'].get('caption', ''),
                            'source': ele_dict['metadata'].get('source', ''),
                            'image_path': ele_dict['metadata'].get('image_path', ''),
                            'page_number': ele_dict['metadata'].get('page_number', '')
                        }
                    )
                    multimodal_documents.append(document)
                else:
                    doc_type = 'Text'
                    text_elements.append(ele)
                
            # Chunking text by title
            text_chunks = chunk_by_title(
                text_elements,
                max_characters=10000,
                combine_text_under_n_chars=500,
            )
            # Turn elements to Documents
            text_documents = []
            for text in text_chunks:
                text = text.to_dict()
                text_documents.append(
                    Document(
                        page_content=text.get('text', ''),
                        metadata={
                            'type': text.get('type', ''),
                            'caption': text.get('caption', ''),
                            'id': text.get('element_id', ''),
                            'source': text['metadata'].get('source', ''),
                            'image_path': text['metadata'].get('image_path', ''),
                            'page_number': text['metadata'].get('page_number', '')
                        }
                    )
                )

            # Split further with RecursiveCharacterTextSplitter
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1500,  # Target size for each chunk
                chunk_overlap=0, # Overlap between consecutive chunks to maintain context
                separators=['\n\n', '\n']
            )
            text_documents = text_splitter.split_documents(text_documents)
            
            documents = text_documents + multimodal_documents
            logger.info(f"Loaded {len(documents)} documents from {file_name}")

            return documents
        except Exception as e:
            logger.error(f"Error executing load_documents for {file_path}: {e}")
            raise e 

    def add_documents(
        self,
        collection_name: str,
        chunks: List[Document],
    ):
        """
        Insert document chunks into Weaviate collection.
        
        Returns:
            None
        """
        self._get_client()

        collection = self._client.collections.get(collection_name)
        
        with collection.batch.dynamic() as batch:
            for doc in chunks:
                properties = {
                    'text': doc.page_content,
                    'chunk_id': doc.metadata.get('id', ''),
                    'type': doc.metadata.get('type', ''),
                    'caption': doc.metadata.get('caption', ''),
                    'source': doc.metadata.get('source', ''),
                    'image_path': doc.metadata.get('image_path', ''),
                    'page_number': doc.metadata.get('page_number', ''),
                }   
                
                batch.add_object(
                    properties=properties,
                    uuid=generate_uuid5(doc.page_content)
                )
        
        failed = len(collection.batch.failed_objects)

        if failed > 0:
            logger.warning(f"Ingestion finished with {failed} failed objects.")
        else:
            logger.info(f"Successfully ingested {len(chunks)} chunks into {collection_name}.")
    
    def ingest(
        self,
        file_name: str,
        collection_name: str = "",
    ):
        """
        Main ingestion pipeline.
        Returns: None
        """
        logger.info(f"Starting ingestion for collection: {collection_name}")
        
        try:
            # Step 1: Preprocess documents
            logger.info(f"STEP 1: Loading documents from {file_name}")
            documents = self.preprocess_documents(str(file_name), collection_name)

            # Step 2: Add documents into Weaviate
            logger.info(f"STEP 2: Adding {len(documents)} documents into Weaviate {collection_name}") 
            self.add_documents(collection_name, documents)

        except Exception as e:
            logger.exception("Ingestion failed")


# NEW update chunking strategy:
1. Add caption to table, image chunks
2. Chunking size reduce from 3000 -> 1500 character by RecursiveCharacterTextSplitter
3. Improve ingestion runtime (original 50s-60s) 

In [ ]:
file_name = "Attention is all you need.pdf"
collection_name = "TestNewChunking"


In [ ]:
from nltk import extract
from src.config import get_settings
settings = get_settings()

file_path = settings.base_dir / "data" / "raw" / collection_name / file_name

# Folder store processed images and tables (per-collection folder)
processed_folder_path = settings.base_dir / "data" / "processed" / collection_name / file_name.split('.')[0]

In [ ]:
elements_first = partition_pdf(
    filename=file_path,
    strategy="hi_res",
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=False,
    extract_image_block_output_dir=processed_folder_path
)
logger.info(f"Extracted {len(elements)} elements from {file_path}")


In [ ]:
len(elements_first)

In [ ]:
# Optimization 
elements = partition_pdf(
    filename=file_path,
    strategy="hi_res",
    # hi_res_model_name="yolox_quantized",   # Faster model -> less accuracy extract elements
    pdf_image_dpi=150,                      # Lower resolution (before 200)
    ocr_mode="individual_blocks",           # Skip full-page OCR (not scanned PDF)
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=False,
    extract_image_block_output_dir=processed_folder_path
)

**reducing half of time 50s -> 25s**

In [ ]:
len(elements)

In [ ]:
types = set([ele.to_dict().get('type') for ele in elements])
print(types)

In [ ]:
image_idx = []
for idx, ele in enumerate(elements):
    ele = ele.to_dict()
    if ele['type'] == 'Image':
        print(ele['text'])
        print('-'*50)
        image_idx.append(idx)


In [ ]:
image_idx[-1]

In [ ]:
elements[204].to_dict()['text']

In [ ]:
for idx, ele in enumerate(elements):
    ele = ele.to_dict()
    if ele['type'] not in ['Image', 'Table']:
        print(ele['text'])
        print('-'*50)


In [ ]:
significant = [
    ele for ele in elements 
    if ele.to_dict().get('type') not in ['UncategorizedText', 'Header'] 
    and len(ele.to_dict().get('text', '')) > 2
]

In [ ]:
len(significant)

In [ ]:
from caption_utils import attach_captions

# 2. Attach captions (modifies Image/Table text in-place, removes caption elements)
significant, message = attach_captions(significant)
logger.info(message)

In [ ]:
for ele in significant:
    ele = ele.to_dict()

    if ele.get('type') in ['Image', 'Table']:
        print(ele['metadata']['caption'])
        print('-'*50)

In [ ]:
# 3. Split into text vs multimodal
text_elements = []
multimodal_elements = []
for ele in significant:
    if ele.to_dict().get('type') in ['Image', 'Table']:
        multimodal_elements.append(ele)
    else:
        ele.to_dict()['type'] = 'Text'
        text_elements.append(ele)

In [ ]:
for ele in multimodal_elements:
    if ele.to_dict()['type'] == 'Image':
        print(ele.to_dict()['text'])
        print('-'*50)

In [ ]:
for ele in text_elements:
    ele = ele.to_dict()
    print(ele['text'])
    print('-'*50)

In [ ]:
# Chunking text by title
chunks = chunk_by_title(
    text_elements,
    max_characters=10000,
    combine_text_under_n_chars=500,
)

# Add multimodal elements to chunks
# chunks.extend(multimodal_elements)

In [ ]:
len(chunks)

In [ ]:
for ele in chunks:
    ele = ele.to_dict()
    print(ele['text'])
    print('-'*50)

In [ ]:
service = TestIngestService()

text_chunks = service._unify_documents(chunks)

In [ ]:
len(text_chunks)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Define the text splitter with parameters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  # Target size for each chunk
    chunk_overlap=0, # Overlap between consecutive chunks to maintain context
    separators=['\n\n', '\n']
)

recursive_chunks = text_splitter.split_documents(text_chunks)

In [ ]:
len(recursive_chunks)

In [ ]:

for ele in recursive_chunks:
    print(ele.page_content)
    print('-'*50)

# Done works: 
1. Enrich tables, images with captions
2. Reduce chunking size from 3000 -> 1500 for efficient semantic chunks
3. Reduce ingestion runtime from 50s -> 25s

# Next work: Modify image summarizer that include caption text

In [ ]:
def _summarize_image(chunk: dict) -> Document:
        """Summarize an image chunk using LLM vision."""
        try:

            caption = chunk['metadata']['caption']

            llm = ChatGoogleGenerativeAI(
                model="gemini-2.0-flash-lite",
                temperature=0.1,
                google_api_key=settings.gemini_api_key
            )
            system_message = SystemMessage(content=f"""You are a document analyst preparing content for a semantic search index.

    You are given:
    1. An image extracted from a document
    2. The image's caption: "{caption}"

    Your task is to write a concise, information-dense summary of this image that will be used as the text representation for vector search retrieval.

    **Instructions:**
    - Use the caption as the primary context anchor — it tells you what the image is about.
    - Describe what the image actually shows: diagrams, charts, tables, architectures, equations, workflows, relationships, etc.
    - Extract ALL specific entities: names, labels, numbers, metrics, axis values, legends, annotations, and technical terms visible in the image.
    - Preserve the original terminology exactly as it appears (do not paraphrase technical terms).
    - State the key takeaway or insight the image conveys.
    - If the image contains comparisons or trends, describe them explicitly (e.g., "X outperforms Y by Z%").
    - Write in plain, factual sentences. Do NOT use bullet points or markdown formatting.
    - Do NOT say "the image shows" or "this figure illustrates" — just state the information directly.
    - Keep the summary between 2-5 sentences, prioritizing information density over length.""")

            # Process image to base64
            try: 
                content = []
                img_path = chunk['metadata']['image_path']
                with open(img_path, 'rb') as f:
                    img_base64 = base64.b64encode(f.read()).decode('utf-8')
                    content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                    })
            except Exception as e:
                logger.warning(f"Warning: Could not load image {img_path}: {e}")
            
            # Create messages and invoke LLM
            human_message = HumanMessage(content=content)
            response = llm.invoke([system_message, human_message])

            summary = response.content.strip()
            chunk['text'] = f"{caption}\n\n{summary}"
            
            logger.info(f"Generated summary for image: {img_path}")

            return chunk 

        except Exception as e:
            logger.error(f"Error generating summaries: {e}")
            raise e

In [ ]:
multimodal_docs = []
for ele in multimodal_elements:
    ele = ele.to_dict()
    document = Document(
        page_content=ele.get('text', ''),
        metadata={
            'type': ele.get('type', ''),
            'id': ele.get('element_id', ''),
            'caption': ele['metadata'].get('caption', ''),
            'source': ele['metadata'].get('source', ''),
            'image_path': ele['metadata'].get('image_path', ''),
            'page_number': ele['metadata'].get('page_number', '')
        }
    )
    multimodal_docs.append(document)
        
len(multimodal_docs)

In [ ]:
for doc in multimodal_docs:
    print(doc.metadata['caption'])
    print('-'*100)
    

In [ ]:
summarized_images = []
for ele in multimodal_elements:
    ele = ele.to_dict()
    if ele['type'] == 'Image':
        ele = _summarize_image(ele)
        summarized_images.append(ele)


In [ ]:
for image in summarized_images:
    print(image['text'])
    print('-'*50)

In [ ]:
multimodal_docs = []
for ele in summarized_images:
    document = Document(
        page_content=ele.get('text', ''),
        metadata={
            'type': ele.get('type', ''),
            'id': ele.get('element_id', ''),
            'caption': ele['metadata'].get('caption', ''),
            'source': ele['metadata'].get('source', ''),
            'image_path': ele['metadata'].get('image_path', ''),
            'page_number': ele['metadata'].get('page_number', '')
        }
    )
    multimodal_docs.append(document)
        
len(multimodal_docs)

In [ ]:
FILE_NAME = "Attention is all you need.pdf"
COLLECTION_NAME = "TestNewChunking"
service = TestIngestService()
documents = service.preprocess_documents(FILE_NAME, COLLECTION_NAME)

In [ ]:
len(documents)

In [ ]:
for document in documents:
    print(document.page_content)
    print("-"*80)